# 05 - RAG Completo com Claude
> Sistema Q&A com citacoes, confianca e avaliacao

**Modulo:** `06_rag` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import chromadb, hashlib
from sentence_transformers import SentenceTransformer
from dataclasses import dataclass
from typing import List

model=SentenceTransformer('all-MiniLM-L6-v2')

@dataclass
class RespostaRAG:
    pergunta:str; resposta:str; fontes:List[str]; confianca:float

class SistemaQA:
    def __init__(self,nome):
        self.col=chromadb.Client().get_or_create_collection(nome)

    def indexar(self,docs):
        ids=[hashlib.md5(d.encode()).hexdigest()[:8]for d in docs]
        self.col.upsert(ids=ids,embeddings=model.encode(docs).tolist(),documents=docs)
        print(f'{len(docs)} docs indexados')

    def perguntar(self,q,n=3,limiar=0.35):
        res=self.col.query(query_embeddings=model.encode([q]).tolist(),n_results=n)
        pares=[(d,1-s)for d,s in zip(res['documents'][0],res['distances'][0])if 1-s>=limiar]
        if not pares: return RespostaRAG(q,'Nao encontrei informacao relevante.',[],0.0)
        docs,scores=zip(*pares)
        ctx='\n'.join(f'[Fonte {i+1}]: {d}'for i,d in enumerate(docs))
        resp=ask(f'Use as fontes abaixo. Cite com [Fonte N].\n{ctx}\n\nPergunta: {q}',model=HAIKU)
        return RespostaRAG(q,resp,list(docs),float(sum(scores)/len(scores)))

qa=SistemaQA('kb')
qa.indexar(['Suporte funciona seg-sex 9h-18h.',
            'SLA: 4h bugs criticos, 24h solicitacoes normais.',
            'Backups diarios as 2h, retidos 30 dias.'])

r=qa.perguntar('Qual o horario do suporte?')
print(f'Resposta: {r.resposta}')
print(f'Confianca: {r.confianca:.0%}')

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
